Matriz Fundamental – Ratón y Comida


## Objetivo

Resolver analíticamente y mediante simulación el siguiente problema:

¿Cuál es la probabilidad de que el ratón, iniciando en la casilla 0, alcance la comida?

El problema será modelado mediante cadenas de Markov absorbentes y matriz fundamental.



# 1. Descripción del problema

El ratón puede desplazarse entre las casillas del tablero.

Estados:

- Estado 7: comida (absorción exitosa)
- Estado 8: descarga eléctrica (shock, absorción fallida)

El ratón inicia en la casilla 0.

Supondremos que:

- En cada movimiento el ratón elige una dirección válida con igual probabilidad.
- El ratón únicamente puede moverse a casillas vecinas conectadas.
- Los estados 7 y 8 son absorbentes.



# 2. Modelado como cadena de Markov

Definimos el espacio de estados:

$$
E = \{0,1,2,3,4,5,6,7,8\}
$$

Los estados absorbentes son:

- 7 (comida)
- 8 (shock)

Los demás estados son transitorios.


In [1]:
import numpy as np
import pandas as pd


# 3. Grafo de conexiones

Del diagrama se obtienen las siguientes conexiones:

- 0 ↔ 1
- 0 ↔ 2
- 1 ↔ 3
- 1 ↔ 7
- 2 ↔ 3
- 2 ↔ 8
- 3 ↔ 4
- 3 ↔ 5
- 4 ↔ 6
- 5 ↔ 6


In [2]:
# Diccionario de vecinos

vecinos = {
    0: [1,2],
    1: [0,3,7],
    2: [0,3,8],
    3: [1,2,4,5],
    4: [3,6],
    5: [3,6],
    6: [4,5],
    7: [7],   # absorbente comida
    8: [8]    # absorbente shock
}


# 4. Construcción de la matriz de transición

La matriz de transición \(P\) contiene las probabilidades:

$$
P_{ij} = P(X_{n+1}=j \mid X_n=i)
$$


In [3]:
N = 9

P = np.zeros((N,N))

for estado in range(N):

    # Estados absorbentes
    if estado in [7,8]:
        P[estado, estado] = 1
        continue

    posibles = vecinos[estado]
    prob = 1 / len(posibles)

    for siguiente in posibles:
        P[estado, siguiente] = prob

In [4]:
pd.DataFrame(P)

,0,1,2,3,4,5,6,7,8
0,0.000000,0.50,0.50,0.000000,0.00,0.00,0.0,0.000000,0.000000
1,0.333333,0.00,0.00,0.333333,0.00,0.00,0.0,0.333333,0.000000
2,0.333333,0.00,0.00,0.333333,0.00,0.00,0.0,0.000000,0.333333
3,0.000000,0.25,0.25,0.000000,0.25,0.25,0.0,0.000000,0.000000
4,0.000000,0.00,0.00,0.500000,0.00,0.00,0.5,0.000000,0.000000
5,0.000000,0.00,0.00,0.500000,0.00,0.00,0.5,0.000000,0.000000
6,0.000000,0.00,0.00,0.000000,0.50,0.50,0.0,0.000000,0.000000
7,0.000000,0.00,0.00,0.000000,0.00,0.00,0.0,1.000000,0.000000
8,0.000000,0.00,0.00,0.000000,0.00,0.00,0.0,0.000000,1.000000



# 5. Forma canónica

Ordenaremos los estados:

$$
0,1,2,3,4,5,6 \mid 7,8
$$

Entonces:

$$
P =
\begin{pmatrix}
Q & R \\
0 & I
\end{pmatrix}
$$


In [5]:
# Estados transitorios
Q = P[:7,:7]

# Transiciones a absorbentes
R = P[:7,7:]


# 6. Matriz fundamental

La matriz fundamental se define como:

$$
F = (I-Q)^{-1}
$$

Esta matriz permite calcular probabilidades de absorción.


In [6]:
I = np.eye(Q.shape[0])

F = np.linalg.inv(I - Q)

F

array([[2. , 1.5, 1.5, 2. , 1. , 1. , 1. ],
       [1. , 2. , 1. , 2. , 1. , 1. , 1. ],
       [1. , 1. , 2. , 2. , 1. , 1. , 1. ],
       [1. , 1.5, 1.5, 4. , 2. , 2. , 2. ],
       [1. , 1.5, 1.5, 4. , 3.5, 2.5, 3. ],
       [1. , 1.5, 1.5, 4. , 2.5, 3.5, 3. ],
       [1. , 1.5, 1.5, 4. , 3. , 3. , 4. ]])


# 7. Probabilidades de absorción

Las probabilidades de absorción se obtienen mediante:

$$
B = FR
$$

Donde:

- Primera columna: probabilidad de terminar en comida (estado 7)
- Segunda columna: probabilidad de terminar en shock (estado 8)


In [7]:
B = F @ R

B_df = pd.DataFrame(
    B,
    columns=['Comida (7)', 'Shock (8)'],
    index=[0,1,2,3,4,5,6]
)

B_df

,Comida (7),Shock (8)
0,0.500000,0.500000
1,0.666667,0.333333
2,0.333333,0.666667
3,0.500000,0.500000
4,0.500000,0.500000
5,0.500000,0.500000
6,0.500000,0.500000



# 8. Resultado analítico

La probabilidad buscada es la probabilidad de absorberse en el estado 7 iniciando desde el estado 0.


In [8]:
prob_comida = B[0,0]

print('Probabilidad de alcanzar la comida iniciando en 0:')
print(prob_comida)

Probabilidad de alcanzar la comida iniciando en 0:
0.4999999999999999



# 9. Interpretación teórica

La matriz:

$$
B = FR
$$

contiene las probabilidades de absorción.

La entrada:

$$
B_{ij}
$$

representa la probabilidad de ser absorbido en el estado absorbente $j$ iniciando desde el estado transitorio $i$.

Por tanto:

$$
B_{0,0}
$$

es la probabilidad de que el ratón llegue a la comida comenzando en la casilla 0.



# 10. Simulación Monte Carlo

Ahora verificaremos el resultado mediante simulación.


In [9]:
from random import choice
from statistics import mean

In [10]:
def simular():

    estado = 0

    while estado not in [7,8]:

        estado = choice(vecinos[estado])

    # Éxito si llega a comida
    if estado == 7:
        return 1
    else:
        return 0

In [11]:
# Número de simulaciones
M = 100000

resultados = []

for _ in range(M):
    resultados.append(simular())

prob_simulada = mean(resultados)

print('Probabilidad simulada:')
print(prob_simulada)

Probabilidad simulada:
0.49824



# 11. Comparación de resultados


In [12]:
print('Resultado analítico :', prob_comida)
print('Resultado simulado  :', prob_simulada)

Resultado analítico : 0.4999999999999999
Resultado simulado  : 0.49824



# 12. Histograma de simulación


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(resultados, bins=2)

plt.xticks([0,1], ['Shock','Comida'])

plt.xlabel('Resultado')
plt.ylabel('Frecuencia')
plt.title('Resultados de la simulación')

plt.show()


# 13. Conclusión

El problema se modeló mediante una cadena de Markov absorbente.

Utilizando la matriz fundamental:

$$
F=(I-Q)^{-1}
$$

y la matriz de absorción:

$$
B=FR
$$

fue posible calcular la probabilidad de que el ratón alcance la comida iniciando desde la casilla 0.

Posteriormente, el resultado fue verificado mediante simulación Monte Carlo, obteniendo un valor muy cercano al resultado teórico.
